# 04 – Indicateur d'artificialisation

Dans le notebook 03, la comparaison des **surfaces totales** de bâti entre années s'est révélée fragile : de petites différences d'acquisition entre les images suffisent à faire varier le total de plusieurs km², même dans des quartiers qui n'ont pas changé.

On mesure donc plutôt les **flux** pixel par pixel entre deux dates :
- **gains** : pixels non bâtis à la première date et bâtis à la seconde (artificialisation) ;
- **pertes** : pixels bâtis puis non bâtis. En réalité ce flux est quasi nul (on ne « désartificialise » presque jamais), donc les pertes détectées donnent un **ordre de grandeur du bruit** de la méthode.

Pour limiter les faux changements, trois filtres :
1. exclusion des pixels sans donnée (nuages masqués) **et d'une bande autour**, où le modèle est perturbé par le bord du masque ;
2. **seuil de confiance** : le modèle doit être sûr de lui aux deux dates ;
3. **surface minimale** : on ne garde que les taches de changement d'au moins 0,5 ha (un chantier fait rarement moins), ce qui élimine le bruit pixel par pixel.

## 1. Chargement des cartes

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
import cv2

DATA_DIR = Path("data_saclay")
RES_DIR = Path("resultats")
CARTES_DIR = RES_DIR / "cartes"

NOMS_CLASSES = ["Bâti", "Forêt / arbres", "Cultures", "Prairies / autre", "Eau"]
BATI = 0
NODATA = 255
HA_PIXEL = 10 * 10 / 1e4      # 1 pixel = 0,01 ha

SEUIL_CONFIANCE = 0.7         # proba max minimale aux deux dates
SURFACE_MIN_HA = 0.5          # taille minimale d'une tache de changement
MARGE_NODATA = 3              # pixels exclus autour des zones sans donnée

cartes, confiance = {}, {}
for annee in ["2017", "2021", "2025"]:
    with rasterio.open(CARTES_DIR / f"occupation_{annee}.tif") as src:
        cartes[annee] = src.read(1)
        profil = src.profile
    with rasterio.open(CARTES_DIR / f"confiance_{annee}.tif") as src:
        confiance[annee] = src.read(1)

H, W = cartes["2017"].shape
print(f"Cartes : {W} x {H} pixels ({W * H * HA_PIXEL / 100:.0f} km²)")

## 2. Calcul des flux entre deux dates

In [ ]:
def masque_valide(a, b):
    nodata = (cartes[a] == NODATA) | (cartes[b] == NODATA)
    noyau = np.ones((2 * MARGE_NODATA + 1, 2 * MARGE_NODATA + 1), np.uint8)
    nodata = cv2.dilate(nodata.astype(np.uint8), noyau).astype(bool)
    return ~nodata


def filtrer_surface(masque, surface_min_ha):
    # supprime les taches (8-connexité) plus petites que surface_min_ha
    n, etiquettes, stats, _ = cv2.connectedComponentsWithStats(masque.astype(np.uint8), connectivity=8)
    garder = stats[:, cv2.CC_STAT_AREA] * HA_PIXEL >= surface_min_ha
    garder[0] = False  # fond
    return garder[etiquettes]


def flux(a, b, seuil=SEUIL_CONFIANCE, surface_min=SURFACE_MIN_HA):
    valide = masque_valide(a, b)
    sur = (confiance[a] >= seuil) & (confiance[b] >= seuil)
    bati_a, bati_b = cartes[a] == BATI, cartes[b] == BATI

    gains_brut = valide & ~bati_a & bati_b
    pertes_brut = valide & bati_a & ~bati_b
    gains = filtrer_surface(gains_brut & sur, surface_min)
    pertes = filtrer_surface(pertes_brut & sur, surface_min)
    return {
        "valide": valide, "gains_brut": gains_brut, "pertes_brut": pertes_brut,
        "gains": gains, "pertes": pertes,
        "bati_initial_ha": float((bati_a & valide).sum() * HA_PIXEL),
    }


def resume(a, b, f):
    g, p = f["gains"].sum() * HA_PIXEL, f["pertes"].sum() * HA_PIXEL
    gb, pb = f["gains_brut"].sum() * HA_PIXEL, f["pertes_brut"].sum() * HA_PIXEL
    duree = int(b) - int(a)
    return {
        "periode": f"{a}-{b}", "duree_ans": duree,
        "gains_bruts_ha": float(gb), "pertes_brutes_ha": float(pb),
        "gains_ha": float(g), "pertes_ha": float(p), "net_ha": float(g - p),
        "net_ha_par_an": float((g - p) / duree),
        "taux_artificialisation": float((g - p) / f["bati_initial_ha"]),
        "part_zone_valide": float(f["valide"].mean()),
    }


periodes = [("2017", "2021"), ("2021", "2025"), ("2017", "2025")]
resultats, flux_calcules = [], {}
for a, b in periodes:
    flux_calcules[(a, b)] = flux(a, b)
    resultats.append(resume(a, b, flux_calcules[(a, b)]))

print(f"{'Période':10s} {'Gains bruts':>12s} {'Pertes brutes':>14s} {'Gains':>9s} {'Pertes':>9s} {'Net':>9s} {'Net/an':>9s} {'Taux':>7s}")
for r in resultats:
    print(f"{r['periode']:10s} {r['gains_bruts_ha']:10.0f} ha {r['pertes_brutes_ha']:11.0f} ha "
          f"{r['gains_ha']:6.0f} ha {r['pertes_ha']:6.0f} ha {r['net_ha']:6.0f} ha {r['net_ha_par_an']:6.0f} ha "
          f"{r['taux_artificialisation']:7.1%}")

Lecture : les colonnes « brutes » correspondent à la simple comparaison des deux cartes, les autres aux changements filtrés (confiance + surface minimale). Si les pertes filtrées restent faibles devant les gains, le filtrage a bien retiré l'essentiel du bruit.

## 3. Origine des surfaces artificialisées (2017 → 2025)

In [ ]:
f = flux_calcules[("2017", "2025")]
origine = cartes["2017"][f["gains"]]
print("Classe en 2017 des surfaces devenues bâties en 2025 :")
origine_ha = {}
for c, nom in enumerate(NOMS_CLASSES):
    if c == BATI:
        continue
    ha = (origine == c).sum() * HA_PIXEL
    origine_ha[nom] = float(ha)
    print(f"  {nom:18s} {ha:8.0f} ha  ({ha / max(f['gains'].sum() * HA_PIXEL, 1e-9):5.1%})")

# Matrice de transition complète (pixels valides et sûrs aux deux dates), en hectares
sur = f["valide"] & (confiance["2017"] >= SEUIL_CONFIANCE) & (confiance["2025"] >= SEUIL_CONFIANCE)
mat = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        mat[i, j] = ((cartes["2017"] == i) & (cartes["2025"] == j) & sur).sum() * HA_PIXEL
print("\nMatrice de transition 2017 (lignes) -> 2025 (colonnes), en ha :")
print(f"{'':18s}" + "".join(f"{n[:10]:>12s}" for n in NOMS_CLASSES))
for i, nom in enumerate(NOMS_CLASSES):
    print(f"{nom:18s}" + "".join(f"{v:12.0f}" for v in mat[i]))

## 4. Sensibilité aux paramètres de filtrage

Le résultat dépend des seuils choisis : on fait varier le seuil de confiance et la surface minimale pour voir si l'ordre de grandeur reste stable.

In [ ]:
seuils = [0.5, 0.6, 0.7, 0.8, 0.9]
surfaces = [0.0, 0.25, 0.5, 1.0]
sens = np.zeros((len(surfaces), len(seuils), 2))
for i, s_min in enumerate(surfaces):
    for j, seuil in enumerate(seuils):
        r = resume("2017", "2025", flux("2017", "2025", seuil, s_min))
        sens[i, j] = r["gains_ha"], r["pertes_ha"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for i, s_min in enumerate(surfaces):
    axes[0].plot(seuils, sens[i, :, 0], "o-", label=f"surface min. {s_min} ha")
    axes[1].plot(seuils, sens[i, :, 0] - sens[i, :, 1], "o-", label=f"surface min. {s_min} ha")
axes[0].set_title("Gains de bâti 2017 → 2025 (ha)")
axes[1].set_title("Artificialisation nette 2017 → 2025 (ha)")
for ax in axes:
    ax.set_xlabel("Seuil de confiance")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RES_DIR / "sensibilite_filtrage.png", dpi=120)
plt.show()

## 5. Cartes des changements

In [ ]:
def fond_rgb(annee):
    with rasterio.open(DATA_DIR / f"s2_{annee}.tif") as src:
        img = np.moveaxis(src.read([3, 2, 1]).astype("float32"), 0, -1)
    bornes = np.percentile(img[img > 0], [2, 98])
    gris = np.clip((img - bornes[0]) / (bornes[1] - bornes[0]), 0, 1).mean(-1)
    return np.dstack([gris] * 3) * 0.6 + 0.2   # fond gris éclairci


def carte_changements(a, b, fenetre=None, ax=None, titre=""):
    f = flux_calcules[(a, b)]
    img = fond_rgb(b)
    img[~f["valide"]] = [1, 1, 1]
    img[f["gains"]] = [0.84, 0.10, 0.11]
    img[f["pertes"]] = [0.17, 0.48, 0.71]
    if fenetre is not None:
        img = img[fenetre]
    ax.imshow(img, interpolation="nearest")
    ax.plot([], [], "s", color=(0.84, 0.10, 0.11), label="Devenu bâti")
    ax.plot([], [], "s", color=(0.17, 0.48, 0.71), label="N'est plus bâti (bruit probable)")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_title(titre)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(12, 11))
carte_changements("2017", "2025", ax=ax, titre="Changements de bâti 2017 → 2025 (filtrés)")
plt.tight_layout()
plt.savefig(RES_DIR / "carte_changements.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Zoom sur le plateau de Saclay (même fenêtre que dans le notebook 03)
from rasterio.warp import transform

with rasterio.open(CARTES_DIR / "occupation_2017.tif") as src:
    xs, ys = transform("EPSG:4326", src.crs, [2.17], [48.712])
    ligne, col = src.index(xs[0], ys[0])
T = 400
zone = (slice(max(ligne - T // 2, 0), ligne + T // 2), slice(max(col - T // 2, 0), col + T // 2))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, (a, b) in zip(axes, periodes):
    carte_changements(a, b, fenetre=zone, ax=ax, titre=f"{a} → {b}")
plt.tight_layout()
plt.savefig(RES_DIR / "zoom_changements.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Validation visuelle

Aucune donnée de référence indépendante n'est disponible pour 2017 et 2025, donc on vérifie à l'œil un échantillon de taches « devenu bâti » sur les images 2017 / 2025.

Les 30 taches sont tirées **au hasard, avec une probabilité proportionnelle à leur surface** : cela revient à vérifier des hectares tirés au hasard, ce qui permet d'estimer la part de la surface détectée qui correspond à de vrais changements (précision). Pour chaque vignette (≈ 600 m de côté, tache entourée en rouge), on note :
- `v` : vrai changement (construction, chantier, nouvelle route…)
- `f` : faux changement (rien n'a changé, ou changement qui n'est pas de l'artificialisation : moisson, coupe d'arbres…)
- `?` : impossible à dire

In [ ]:
N_VALID = 30
TAILLE = 60   # demi-côté des vignettes en pixels (60 px = 600 m)

f = flux_calcules[("2017", "2025")]
n, etiq, stats, centres = cv2.connectedComponentsWithStats(f["gains"].astype(np.uint8), connectivity=8)
aires = stats[1:, cv2.CC_STAT_AREA].astype(float)        # composante 0 = fond
rng = np.random.default_rng(0)
choix = rng.choice(np.arange(1, n), size=min(N_VALID, n - 1), replace=False, p=aires / aires.sum())
print(f"{n - 1} taches détectées, {len(choix)} tirées pour la validation")


def rgb_complet(annee):
    with rasterio.open(DATA_DIR / f"s2_{annee}.tif") as src:
        img = np.moveaxis(src.read([3, 2, 1]).astype("float32"), 0, -1)
    bornes = np.percentile(img[img > 0], [2, 98])
    return np.clip((img - bornes[0]) / (bornes[1] - bornes[0]), 0, 1)

rgb = {a: rgb_complet(a) for a in ["2017", "2025"]}

echantillon = []
fig, axes = plt.subplots(6, 10, figsize=(25, 16))
for k, comp in enumerate(choix):
    cx, cy = centres[comp].astype(int)
    y0, y1 = max(cy - TAILLE, 0), min(cy + TAILLE, H)
    x0, x1 = max(cx - TAILLE, 0), min(cx + TAILLE, W)
    tache = etiq[y0:y1, x0:x1] == comp
    contour = tache & ~cv2.erode(tache.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    classe_origine = np.bincount(cartes["2017"][etiq == comp], minlength=256)[:5].argmax()
    echantillon.append({"id": k, "surface_ha": float(stats[comp, cv2.CC_STAT_AREA] * HA_PIXEL),
                        "origine": NOMS_CLASSES[classe_origine], "centre_px": [int(cx), int(cy)]})
    for j, annee in enumerate(["2017", "2025"]):
        ax = axes.ravel()[2 * k + j]
        vignette = rgb[annee][y0:y1, x0:x1].copy()
        vignette[contour] = [1, 0, 0]
        ax.imshow(vignette, interpolation="nearest")
        ax.set_title(f"#{k} {annee}" + (f" – {echantillon[-1]['surface_ha']:.1f} ha, {NOMS_CLASSES[classe_origine]}" if j == 0 else ""),
                     fontsize=8)
        ax.axis("off")
for ax in axes.ravel()[2 * len(choix):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(RES_DIR / "validation_vignettes.png", dpi=110, bbox_inches="tight")
plt.show()

Après avoir regardé les vignettes, remplir `VERDICTS` (un caractère par vignette, dans l'ordre #0, #1, …) puis exécuter la cellule suivante.

In [ ]:
VERDICTS = ""   # ex. "vvfv?vvf..." : 30 caractères, v = vrai, f = faux, ? = indécis

assert len(VERDICTS) == len(echantillon), f"{len(VERDICTS)} verdicts pour {len(echantillon)} vignettes"
for e, v in zip(echantillon, VERDICTS):
    e["verdict"] = v

juges = [e for e in echantillon if e["verdict"] in "vf"]
vrais = [e for e in juges if e["verdict"] == "v"]
precision = len(vrais) / len(juges)
# intervalle de confiance à 95 % (approximation normale)
ic = 1.96 * np.sqrt(precision * (1 - precision) / len(juges))
print(f"Précision estimée : {len(vrais)}/{len(juges)} = {precision:.0%} (IC 95 % : {max(precision - ic, 0):.0%} – {min(precision + ic, 1):.0%})")
print(f"Indécis : {sum(e['verdict'] == '?' for e in echantillon)}")

print("\nPar classe d'origine :")
for nom in NOMS_CLASSES[1:]:
    sous = [e for e in juges if e["origine"] == nom]
    if sous:
        print(f"  {nom:18s} {sum(e['verdict'] == 'v' for e in sous)}/{len(sous)} vrais")

r = resume("2017", "2025", f)
gains_corriges = r["gains_ha"] * precision
print(f"\nGains 2017-2025 corrigés de la précision : {r['gains_ha']:.0f} ha x {precision:.0%} = {gains_corriges:.0f} ha "
      f"(soit {gains_corriges / 8:.0f} ha/an)")

with open(RES_DIR / "validation.json", "w", encoding="utf-8") as fp:
    json.dump({"precision": precision, "ic95": [max(precision - ic, 0), min(precision + ic, 1)],
               "gains_corriges_ha": gains_corriges, "echantillon": echantillon}, fp, indent=2, ensure_ascii=False)

## 7. Sauvegarde

In [ ]:
# Raster des changements 2017 -> 2025 : 0 = pas de changement, 1 = devenu bâti, 2 = n'est plus bâti, 255 = non valide
f = flux_calcules[("2017", "2025")]
changements = np.zeros((H, W), np.uint8)
changements[f["gains"]] = 1
changements[f["pertes"]] = 2
changements[~f["valide"]] = 255
profil.update(dtype="uint8", nodata=255, count=1, compress="deflate")
with rasterio.open(CARTES_DIR / "changements_2017_2025.tif", "w", **profil) as dst:
    dst.write(changements, 1)

indicateurs = {
    "parametres": {"seuil_confiance": SEUIL_CONFIANCE, "surface_min_ha": SURFACE_MIN_HA, "marge_nodata_px": MARGE_NODATA},
    "flux": resultats,
    "origine_artificialisation_2017_2025_ha": origine_ha,
    "matrice_transition_2017_2025_ha": {"classes": NOMS_CLASSES, "valeurs": mat.round(1).tolist()},
}
with open(RES_DIR / "indicateurs.json", "w", encoding="utf-8") as fp:
    json.dump(indicateurs, fp, indent=2, ensure_ascii=False)
print(json.dumps(resultats, indent=2, ensure_ascii=False))

## Observations

*(à compléter après exécution : surface artificialisée nette et par an, part issue des cultures, localisation des principaux changements, niveau de bruit estimé par les pertes, stabilité du résultat selon les seuils)*